# Challenge 2: RAG System with BigQuery

This notebook implements Retrieval Augmented Generation (RAG) using BigQuery ML.

## Architecture
1. Load Aurora Bay FAQ data into BigQuery
2. Generate embedding vectors for each FAQ using BigQuery ML
3. At query time, embed the user's question and run a vector search to find relevant FAQs
4. Pass the matching FAQs as context to Gemini to generate an accurate answer

In [1]:
!pip install -q google-cloud-bigquery google-cloud-bigquery-connection \
               google-cloud-aiplatform vertexai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 93.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.8/131.8 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.5/320.5 kB 31.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colabsqlviz 0.2.13 requires protobuf<7.0.0,>=6.31.1, but you have protobuf 5.29.6 which is incompatible.
google-adk 1.29.0 requires google-cloud-aiplatform[agent-engines]<2.0.0,>=1.132.0, but you have google-cloud-aiplatform 1.71.1 which is incompatible.
wandb 0.27.0 requires click>=8.2.0, but you have click 8.1.8 which is incompatible.


## Configuration

In [2]:
PROJECT_ID  = "qwiklabs-gcp-04-a13cec945fb9"
BQ_LOCATION = "US"           # BigQuery multi-region — required for remote models
DATASET_ID  = "AuroraBay"
RAW_TABLE   = "aurora_bay_faqs"
EMB_TABLE   = "aurora_bay_faqs_embedded"
EMB_MODEL   = "EmbeddingModel"
CONN_ID     = "embedding_conn"
GCS_URI     = "gs://labs.roitraining.com/aurora-bay-faqs/aurora-bay-faqs.csv"

GEMINI_MODEL    = "gemini-2.5-flash"
VERTEX_LOCATION = "us-central1"

print("Configuration set.")

Configuration set.


## Authenticate and Initialize Clients

In [3]:
import vertexai
from vertexai.generative_models import GenerativeModel, GenerationConfig
from google.cloud import bigquery
from google.cloud import bigquery_connection_v1 as bq_conn
import google.auth

credentials, project = google.auth.default()

bq_client   = bigquery.Client(project=PROJECT_ID, credentials=credentials)
conn_client = bq_conn.ConnectionServiceClient(credentials=credentials)

vertexai.init(project=PROJECT_ID, location=VERTEX_LOCATION, credentials=credentials)

print("Clients initialized.")

Clients initialized.


## Step 1: Create BigQuery Dataset and Load FAQ Data

We create the AuroraBay dataset if it doesn't exist, then load the FAQ CSV
directly from Google Cloud Storage into a BigQuery table.

In [4]:
# Create dataset if it doesn't exist
dataset_ref = bigquery.Dataset(f"{PROJECT_ID}.{DATASET_ID}")
dataset_ref.location = BQ_LOCATION

try:
    bq_client.create_dataset(dataset_ref)
    print(f"✅ Dataset {DATASET_ID} created.")
except Exception:
    print(f"✅ Dataset {DATASET_ID} already exists.")

# Load CSV with explicit schema so columns are named question / answer
job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,          # skip the header row
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
    schema=[
        bigquery.SchemaField("question", "STRING"),
        bigquery.SchemaField("answer",   "STRING"),
    ],
)

load_job = bq_client.load_table_from_uri(
    GCS_URI,
    f"{PROJECT_ID}.{DATASET_ID}.{RAW_TABLE}",
    job_config=job_config,
)
load_job.result()

table = bq_client.get_table(f"{PROJECT_ID}.{DATASET_ID}.{RAW_TABLE}")
print(f"✅ Loaded {table.num_rows} rows into {DATASET_ID}.{RAW_TABLE}")
print(f"   Schema: {[f.name for f in table.schema]}")

✅ Dataset AuroraBay already exists.
✅ Loaded 50 rows into AuroraBay.aurora_bay_faqs
   Schema: ['question', 'answer']


## Step 2: Create BigQuery Connection for Remote Models

BigQuery ML needs a Cloud Resource connection to call Vertex AI embedding models.
We create the connection, then grant its service account the Vertex AI User role. I added in a slight wait since I was having issues with the account not existing before it tried to add the role.

In [5]:
import time
from google.cloud import resourcemanager_v3
from google.iam.v1 import iam_policy_pb2, policy_pb2

# ── Create (or reuse) the Cloud Resource connection ───────────────
# BigQuery ML calls Vertex AI through a Cloud Resource connection. The
# connection owns an auto-generated service account that we grant access to
# Vertex AI below. The US multi-region connection lives in location 'us'.
from google.api_core.exceptions import AlreadyExists

conn_parent = f"projects/{PROJECT_ID}/locations/{BQ_LOCATION.lower()}"
connection  = bq_conn.Connection(cloud_resource=bq_conn.CloudResourceProperties())

try:
    created = conn_client.create_connection(
        parent=conn_parent,
        connection_id=CONN_ID,
        connection=connection,
    )
    service_account = created.cloud_resource.service_account_id
    print(f"✅ Connection {CONN_ID} created.")
except AlreadyExists:
    existing = conn_client.get_connection(
        name=f"{conn_parent}/connections/{CONN_ID}"
    )
    service_account = existing.cloud_resource.service_account_id
    print(f"✅ Connection {CONN_ID} already exists — reusing it.")

print(f"Service account: {service_account}")
print("Waiting 15 seconds for service account to be provisioned...")
time.sleep(15)

# Retry the IAM binding up to 3 times
crm_client = resourcemanager_v3.ProjectsClient(credentials=credentials)
project_name = f"projects/{PROJECT_ID}"

for attempt in range(1, 4):
    try:
        # Get current IAM policy
        policy = crm_client.get_iam_policy(
            request={"resource": project_name}
        )

        # Add the binding
        binding = policy_pb2.Binding(
            role="roles/aiplatform.user",
            members=[f"serviceAccount:{service_account}"],
        )
        policy.bindings.append(binding)

        # Set updated policy
        crm_client.set_iam_policy(
            request={"resource": project_name, "policy": policy}
        )
        print(f"✅ IAM role granted on attempt {attempt}.")
        break

    except Exception as e:
        print(f"  Attempt {attempt} failed: {e}")
        if attempt < 3:
            print("  Retrying in 15 seconds...")
            time.sleep(15)
        else:
            print("❌ Could not grant IAM role automatically.")
            print(f"   Run this manually in Cloud Shell:")
            print(f"   gcloud projects add-iam-policy-binding {PROJECT_ID} \\")
            print(f"     --member='serviceAccount:{service_account}' \\")
            print(f"     --role='roles/aiplatform.user'")

✅ Connection embedding_conn already exists — reusing it.
Service account: bqcx-131669982938-n4st@gcp-sa-bigquery-condel.iam.gserviceaccount.com
Waiting 15 seconds for service account to be provisioned...
✅ IAM role granted on attempt 1.


## Step 3: Create the Embedding Model in BigQuery ML

This creates a remote model in BigQuery that points to the Vertex AI
text-embedding-005 model. BigQuery will call this model when we run
ML.GENERATE_EMBEDDING.

In [6]:
create_model_sql = f"""
CREATE OR REPLACE MODEL `{PROJECT_ID}.{DATASET_ID}.{EMB_MODEL}`
REMOTE WITH CONNECTION `{PROJECT_ID}.{BQ_LOCATION}.{CONN_ID}`
OPTIONS (ENDPOINT = 'text-embedding-005')
"""

print("Creating embedding model in BigQuery ML...")
bq_client.query(create_model_sql).result()
print(f"✅ Model {EMB_MODEL} created.")

Creating embedding model in BigQuery ML...
✅ Model EmbeddingModel created.


## Step 4: Inspect the FAQ Schema

Before generating embeddings we check the actual column names from the loaded CSV
so we know what to concatenate as the embedding content.

In [7]:
# Preview the first few rows to understand the data structure
preview = bq_client.query(f"""
    SELECT * FROM `{PROJECT_ID}.{DATASET_ID}.{RAW_TABLE}` LIMIT 3
""").result()

print("Schema and sample data:")
rows = list(preview)
for field in preview.schema:
    print(f"  Column: {field.name} ({field.field_type})")

print("\nSample rows:")
for row in rows:
    print(dict(row))

Schema and sample data:
  Column: question (STRING)
  Column: answer (STRING)

Sample rows:
{'question': 'When was Aurora Bay founded?', 'answer': 'Aurora Bay was founded in 1901 by a group of fur traders who recognized the region’s strategic coastal location.'}
{'question': 'What is the population of Aurora Bay?', 'answer': 'Aurora Bay has a population of approximately 3,200 residents, although it can fluctuate seasonally due to temporary fishing and tourism workforces.'}
{'question': 'Where is the Aurora Bay Town Hall located?', 'answer': 'The Town Hall is located at 100 Harbor View Road, in the center of Aurora Bay, close to the main harbor.'}


## Step 5: Generate Embeddings

We concatenate the question and answer into a single content string, then use
ML.GENERATE_EMBEDDING to create a vector for each FAQ. The result — including
the original fields and the embedding vector — is stored in a new table.

This only needs to run once. If you re-run it, WRITE_TRUNCATE replaces the table.

In [8]:
# We combine question + answer so the embedding captures the full context.
# Update column names here if the schema preview above showed different names.
generate_embeddings_sql = f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET_ID}.{EMB_TABLE}` AS
SELECT *
FROM ML.GENERATE_EMBEDDING(
    MODEL `{PROJECT_ID}.{DATASET_ID}.{EMB_MODEL}`,
    (
        SELECT
            CONCAT(question, ' ', answer) AS content,
            question,
            answer
        FROM `{PROJECT_ID}.{DATASET_ID}.{RAW_TABLE}`
    )
)
"""

print("Generating embeddings — this may take 1-2 minutes...")
bq_client.query(generate_embeddings_sql).result()

count = list(bq_client.query(
    f"SELECT COUNT(*) AS n FROM `{PROJECT_ID}.{DATASET_ID}.{EMB_TABLE}`"
).result())[0]["n"]

print(f"✅ Embeddings generated and stored. {count} rows in {EMB_TABLE}.")

Generating embeddings — this may take 1-2 minutes...
✅ Embeddings generated and stored. 50 rows in aurora_bay_faqs_embedded.


## Step 6: Vector Search Function

At query time we:
1. Embed the user's question using the same model
2. Run VECTOR_SEARCH to find the top-k most similar FAQs by cosine similarity
3. Return the matching question/answer pairs as context

In [9]:
def search_faqs(user_question: str, top_k: int = 3) -> list[dict]:
    """
    Embed the user question and find the most semantically similar FAQs
    using BigQuery VECTOR_SEARCH.

    Args:
        user_question: The user's natural language question.
        top_k: Number of FAQ matches to return.

    Returns:
        List of dicts with 'question' and 'answer' keys.
    """
    # Escape single quotes in the question to prevent SQL injection
    safe_question = user_question.replace("'", "\\'")

    vector_search_sql = f"""
    SELECT
        base.question,
        base.answer,
        distance
    FROM VECTOR_SEARCH(
        TABLE `{PROJECT_ID}.{DATASET_ID}.{EMB_TABLE}`,
        'ml_generate_embedding_result',
        (
            SELECT ml_generate_embedding_result, content AS query
            FROM ML.GENERATE_EMBEDDING(
                MODEL `{PROJECT_ID}.{DATASET_ID}.{EMB_MODEL}`,
                (SELECT '{safe_question}' AS content)
            )
        ),
        top_k => {top_k},
        options => '{{"fraction_lists_to_search": 0.5}}'
    )
    ORDER BY distance ASC
    """

    results = bq_client.query(vector_search_sql).result()
    return [dict(row) for row in results]


print("search_faqs() defined.")

search_faqs() defined.


## Step 7: Gemini Answer Generation

We pass the retrieved FAQs as context to Gemini with a prompt that instructs it
to answer only from the provided data. This is the RAG pattern — the model is
grounded in our proprietary data rather than its training data.

In [10]:
SYSTEM_INSTRUCTION = """
You are a helpful assistant for Aurora Bay, Alaska.
Answer questions using only the FAQ information provided to you.
If the answer is not covered by the FAQs, say you don't have that information
and suggest the user contact the Aurora Bay town office directly.
Keep answers friendly, concise, and accurate.
"""

GENERATION_CONFIG = GenerationConfig(
    temperature=0.2,
    max_output_tokens=1024,
)

gemini_model = GenerativeModel(
    model_name=GEMINI_MODEL,
    system_instruction=SYSTEM_INSTRUCTION,
    generation_config=GENERATION_CONFIG,
)


def ask_aurora_bay(user_question: str) -> str:
    """
    Full RAG pipeline:
      1. Vector search BigQuery for relevant FAQs
      2. Build a grounded prompt with the retrieved context
      3. Call Gemini to generate the answer

    Args:
        user_question: The user's question about Aurora Bay.

    Returns:
        Gemini's answer grounded in the FAQ data.
    """
    # Step 1: Retrieve relevant FAQs
    relevant_faqs = search_faqs(user_question, top_k=3)

    if not relevant_faqs:
        return "I couldn't find any relevant information in the Aurora Bay FAQ database."

    # Step 2: Build context block from retrieved FAQs
    context = "\n\n".join([
        f"Q: {faq['question']}\nA: {faq['answer']}"
        for faq in relevant_faqs
    ])

    # Step 3: Build the grounded prompt
    prompt = f"""Use the following Aurora Bay FAQ entries to answer the question below.

--- FAQ Context ---
{context}
--- End Context ---

User Question: {user_question}
"""

    response = gemini_model.generate_content(prompt)
    return response.text


print("ask_aurora_bay() defined.")

ask_aurora_bay() defined.


## Test 1: Question covered by the FAQs

Expected: Gemini answers accurately using retrieved FAQ data.

In [12]:
question = "Where is the Aurora Bay Town Hall located?"
print(f"Question: {question}\n")
print("Answer:", ask_aurora_bay(question))

Question: Where is the Aurora Bay Town Hall located?

Answer: The Aurora Bay Town Hall is located at 100 Harbor View Road, in the center of Aurora Bay, close to the main harbor.


## Test 2: Question not in the FAQs

Expected: Gemini says it doesn't have that information rather than hallucinating.

In [13]:
question = "What is the population of Madison WI?"
print(f"Question: {question}\n")
print("Answer:", ask_aurora_bay(question))

Question: What is the population of Madison WI?

Answer: I don't have that information. Please contact the Aurora Bay town office directly.


## Test 3: Show retrieved context

This cell shows the raw vector search results so you can verify the RAG
retrieval is working correctly before Gemini generates the answer.

In [14]:
question = "What are the hours for the Aurora Bay library?"
print(f"Question: {question}\n")

faqs = search_faqs(question, top_k=3)
print("Top 3 retrieved FAQs:")
for i, faq in enumerate(faqs, 1):
    print(f"\n[{i}] Q: {faq['question']}")
    print(f"    A: {faq['answer']}")
    print(f"    Distance: {faq['distance']:.4f}")

print("\n--- Gemini Answer ---")
print(ask_aurora_bay(question))

Question: What are the hours for the Aurora Bay library?

Top 3 retrieved FAQs:

[1] Q: What are the operating hours of the Aurora Bay Public Library?
    A: The library is open Monday through Friday from 9 AM to 6 PM, and on Saturdays from 10 AM to 4 PM. It’s closed on Sundays and major holidays.
    Distance: 0.3957

[2] Q: Does Aurora Bay have a public library?
    A: Yes. The Aurora Bay Public Library is located on Main Street, next to the town’s post office.
    Distance: 0.5964

[3] Q: Where is the Aurora Bay Town Hall located?
    A: The Town Hall is located at 100 Harbor View Road, in the center of Aurora Bay, close to the main harbor.
    Distance: 0.7751

--- Gemini Answer ---
The Aurora Bay Public Library is open Monday through Friday from 9 AM to 6 PM, and on Saturdays from 10 AM to 4 PM. It is closed on Sundays and major holidays.


## Interactive Chat (Optional)
Running this will start the interactive chat with the Aurora Bay Assistant - I have done a simply run asking a question it should know from the Q/A and then followed it up with one it wouldnt know to verify that flow as well :)

In [15]:
def run_aurora_bay_chat():
    print("Aurora Bay Assistant — type 'exit' to quit.\n")
    while True:
        question = input("You: ").strip()
        if not question:
            continue
        if question.lower() in ("exit", "quit"):
            print("Goodbye!")
            break
        print(f"\nAssistant: {ask_aurora_bay(question)}\n")

run_aurora_bay_chat()

Aurora Bay Assistant — type 'exit' to quit.

You: where is the town hall

Assistant: The Aurora Bay Town Hall is located at 100 Harbor View Road, in the center of Aurora Bay, close to the main harbor.

You: is the post office near the library?

Assistant: Yes, the Aurora Bay Public Library is located on Main Street, next to the town's post office.

You: should I buy bitcoin?

Assistant: I don't have information about whether you should buy Bitcoin. For questions not covered in these FAQs, please contact the Aurora Bay town office directly.

You: exit
Goodbye!


## Summary

| Step | What happens | Tool |
|---|---|---|
| 1 | Load FAQ CSV | BigQuery Load Job (from GCS) |
| 2 | Create remote connection | BigQuery Connection API |
| 3 | Create embedding model | BigQuery ML (text-embedding-005) |
| 4 | Generate & store vectors | ML.GENERATE_EMBEDDING |
| 5 | Search at query time | VECTOR_SEARCH (nearest neighbor) |
| 6 | Generate grounded answer | Gemini 2.5 Flash |

The model only answers from the retrieved FAQ context — it cannot hallucinate
information that isn't in the Aurora Bay proprietary data.